# 9.12 Serving 引擎内部深挖 (调度 / Paged KV / Radix)

> 🕐 预估学习时间：45分钟

vLLM / SGLang 类引擎的核心不是“包一层 API”，而是：**连续批处理调度 + 分页 KV + 前缀树复用**。本节用纯 Python 模拟关键数据结构与调度决策。

深挖点：
- 等待队列 vs 运行队列
- 块分配/回收与碎片
- 前缀匹配命中率
- 抢占与公平性


## 1. 分页 KV 块分配器

逻辑 token 序列映射到非连续物理块，避免预留最大长度造成浪费。


In [ ]:
from dataclasses import dataclass, field


@dataclass
class BlockAllocator:
    n_blocks: int
    free: list = field(init=False)
    table: dict = field(default_factory=dict)  # req -> list[block_id]

    def __post_init__(self):
        self.free = list(range(self.n_blocks))

    def alloc(self, req, n):
        if len(self.free) < n:
            return False
        blocks = [self.free.pop() for _ in range(n)]
        self.table.setdefault(req, []).extend(blocks)
        return True

    def free_req(self, req):
        for b in self.table.pop(req, []):
            self.free.append(b)


alloc = BlockAllocator(16)
print('=== Block Allocator ===')
print('alloc A 4', alloc.alloc('A', 4), 'free', len(alloc.free))
print('alloc B 10', alloc.alloc('B', 10), 'free', len(alloc.free))
print('alloc C 4', alloc.alloc('C', 4))  # should fail
alloc.free_req('A')
print('after free A, alloc C 4', alloc.alloc('C', 4), 'free', len(alloc.free))
print('Key: Paging trades indirection for near-zero fragmentation waste.')


## 2. Continuous batching 调度器

decode 步优先填满 batch；有空位再 prefill 新请求。


In [ ]:
@dataclass
class Req:
    rid: int
    remaining_prefill: int
    remaining_decode: int
    stage: str = 'waiting'


class Scheduler:
    def __init__(self, max_batched_tokens=64):
        self.max_batched_tokens = max_batched_tokens
        self.waiting = []
        self.running = []
        self.done = []
        self.t = 0

    def add(self, req):
        self.waiting.append(req)

    def step(self):
        self.t += 1
        # finish decode slots and free capacity
        still = []
        used = 0
        for r in self.running:
            if r.stage == 'prefill':
                take = min(r.remaining_prefill, self.max_batched_tokens - used)
                r.remaining_prefill -= take
                used += take
                if r.remaining_prefill == 0:
                    r.stage = 'decode'
                still.append(r)
            else:
                if used < self.max_batched_tokens:
                    r.remaining_decode -= 1
                    used += 1
                if r.remaining_decode <= 0:
                    self.done.append(r)
                else:
                    still.append(r)
        self.running = still
        # admit waiting as prefill if space
        while self.waiting and used < self.max_batched_tokens:
            r = self.waiting[0]
            if r.remaining_prefill <= self.max_batched_tokens - used or not self.running:
                self.waiting.pop(0)
                r.stage = 'prefill'
                self.running.append(r)
                used += min(r.remaining_prefill, self.max_batched_tokens - used)
            else:
                break
        return used


sch = Scheduler(max_batched_tokens=16)
for i, (p, d) in enumerate([(30, 4), (8, 10), (12, 6)]):
    sch.add(Req(i, p, d))
utils = []
while len(sch.done) < 3 and sch.t < 40:
    utils.append(sch.step())
print('=== Continuous Batching ===')
print('utilization per step', utils)
print('finish times', [(r.rid, sch.t) for r in sch.done])
print('Key: Mixing prefill/decode under a token budget is the heart of online LLM serving.')


## 3. Radix 前缀复用

多请求共享系统提示/工具前缀时，缓存树节点可复用 KV 块。


In [ ]:
class RadixNode:
    def __init__(self):
        self.children = {}
        self.block_ids = []
        self.hits = 0


class RadixCache:
    def __init__(self):
        self.root = RadixNode()

    def match_or_insert(self, tokens):
        node = self.root
        matched = 0
        for tok in tokens:
            if tok in node.children:
                node = node.children[tok]
                matched += 1
                node.hits += 1
            else:
                break
        # insert rest
        for tok in tokens[matched:]:
            nxt = RadixNode()
            nxt.block_ids = [hash((id(node), tok)) % 10_000]
            node.children[tok] = nxt
            node = nxt
        return matched


cache = RadixCache()
prompts = [
    [1, 2, 3, 4, 5],
    [1, 2, 3, 9, 9],
    [1, 2, 7],
    [8, 8, 8],
]
print('=== Radix Prefix Hits ===')
for p in prompts:
    m = cache.match_or_insert(p)
    print(p, 'matched_prefix_len', m)
print('Key: Higher prefix hit rate → less prefill compute; tree eviction policies matter under memory pressure.')


## 4. 公平性与抢占

长 prefill 可能饿死短交互。可对 waiting 用老化优先级，或限制单步 prefill 长度（chunked prefill）。


In [ ]:
def admit_priority(waiting, now):
    # higher score first: age - beta * prefill_len
    return sorted(waiting, key=lambda r: (now - r.rid) - 0.01 * r.remaining_prefill, reverse=True)


waiting = [Req(0, 100, 5), Req(1, 8, 5), Req(2, 12, 5)]
print('=== Fairness Ordering ===')
print([(r.rid, r.remaining_prefill) for r in admit_priority(waiting, now=10)])
print('Key: Production schedulers explicitly optimize TTFT fairness, not only average throughput.')


## 课后思考题

1. chunked prefill 如何改善尾部 TTFT？代价是什么？
2. 前缀缓存与多租户隔离冲突时如何设计命名空间？
3. 抢占正在 decode 的请求需要保存哪些状态？
4. 如何用线上 trace 回放评估调度策略改动？

---
> 本节是Serving 引擎内部的垂直深挖。建议对照真实训练日志/线上指标复现关键实验，而不是只跑通玩具代码。
